In [ ]:
from ultralytics import YOLO
import cv2
import os

model = YOLO("runs/detect/deteccion_robo/weights/best.pt")

image_path = "robo.jpeg"

results = model.predict(
    source=image_path,
    conf=0.25,
    save=True
)

save_dir = results[0].save_dir
image_name = os.path.basename(image_path)
result_image_path = os.path.join(save_dir, image_name)

img = cv2.imread(result_image_path)

if img is None:
    print("No se pudo leer la imagen:", result_image_path)
else:
    cv2.imshow("Resultado YOLO", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


image 1/1 c:\Users\User\Desktop\EXP4\robo.jpeg: 640x640 1 robo, 96.9ms
Speed: 2.3ms preprocess, 96.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)


In [37]:
from ultralytics import YOLO
import cv2

model = YOLO("runs/detect/deteccion_robo/weights/best.pt")

results = model.predict(
    source="robo3.jpeg",
    conf=0.25
)

annotated_img = results[0].plot()

cv2.imshow("Resultado YOLO", annotated_img)
cv2.waitKey(0)
cv2.destroyAllWindows()


image 1/1 c:\Users\User\Desktop\EXP4\robo3.jpeg: 416x640 1 robo, 67.8ms
Speed: 1.1ms preprocess, 67.8ms inference, 0.6ms postprocess per image at shape (1, 3, 416, 640)


================================

# Interpretabilidad del modelo YOLOv8 mediante Grad-CAM

En este notebook se aplica la técnica Grad-CAM al modelo YOLOv8 entrenado para la detección de actividades sospechosas o robo. El objetivo es visualizar las regiones de la imagen que influyen en la predicción realizada por el modelo.

Este análisis se realiza utilizando el modelo entrenado `best.pt`, por lo que no es necesario volver a entrenar el modelo.

## 1. Importación de librerías

Se importan las librerías necesarias para cargar el modelo YOLOv8, procesar imágenes, trabajar con PyTorch y generar el mapa de calor mediante Grad-CAM.

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import torch.nn as nn

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image

## 2. Carga del modelo entrenado

Se carga el modelo YOLOv8 previamente entrenado utilizando el archivo `best.pt`. Este archivo contiene los pesos del modelo con mejor desempeño durante el entrenamiento.

In [2]:
yolo = YOLO("runs/detect/deteccion_robo/weights/best.pt")

net = yolo.model
net.eval()

for p in net.parameters():
    p.requires_grad_(True)

## 3. Adaptación del modelo YOLO para Grad-CAM

YOLOv8 devuelve una salida más compleja que un modelo de clasificación tradicional, ya que genera cajas, clases y puntuaciones. Por ello, se crea un wrapper para adaptar la salida del modelo y permitir que Grad-CAM pueda procesarla.

In [3]:
class YOLOWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        output = self.model(x)

        if isinstance(output, (tuple, list)):
            output = output[0]

        return output


wrapped_model = YOLOWrapper(net)

## 4. Definición de la clase objetivo

Se define la clase que se desea analizar mediante Grad-CAM. En este caso, se selecciona la clase `robo`, cuyo índice corresponde a `1` según la configuración del dataset.

## 4. Definición de la clase objetivo

Se define la clase que se desea analizar mediante Grad-CAM. En este caso, se selecciona la clase `robo`, cuyo índice corresponde a `1` según la configuración del dataset.

In [4]:
class YOLOClassTarget:
    def __init__(self, class_index):
        self.class_index = class_index

    def __call__(self, model_output):
        """
        Salida esperada del modelo:
        [batch, canales, predicciones]

        Donde los primeros 4 canales corresponden a las coordenadas
        de la caja delimitadora, y los canales restantes corresponden
        a las clases del modelo.
        """

        if model_output.dim() == 3:
            class_scores = model_output[0, 4 + self.class_index, :]

        elif model_output.dim() == 2:
            class_scores = model_output[4 + self.class_index, :]

        else:
            raise ValueError(f"Forma de salida no esperada: {model_output.shape}")

        return class_scores.max()

## 5. Selección de la clase y capa objetivo

Se selecciona la clase `robo` como objetivo del análisis. Además, se define una capa interna del modelo YOLOv8 para generar el mapa de calor. La capa puede modificarse si el resultado visual no es adecuado.

In [5]:
# Clases del modelo:
# 0 = normal
# 1 = robo

class_index = 1
targets = [YOLOClassTarget(class_index)]

target_layers = [net.model[-2]]

# Opciones alternativas:
# target_layers = [net.model[-3]]
# target_layers = [net.model[-4]]

## 6. Lectura y preparación de la imagen

Se carga la imagen que será analizada por Grad-CAM. Luego, se convierte de BGR a RGB, se redimensiona a 640x640 píxeles y se normaliza para que pueda ser procesada por el modelo.

In [6]:
image_path = "robo3.jpeg"

bgr_img = cv2.imread(image_path)

if bgr_img is None:
    raise FileNotFoundError(f"No se pudo leer la imagen: {image_path}")

rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
rgb_img = cv2.resize(rgb_img, (640, 640))

rgb_float = np.float32(rgb_img) / 255.0

## 7. Conversión de la imagen a tensor

La imagen se convierte a tensor para que pueda ser utilizada por PyTorch. Además, se activan los gradientes en la entrada, ya que Grad-CAM necesita calcular la influencia de las regiones de la imagen sobre la predicción.

In [7]:
input_tensor = preprocess_image(
    rgb_float,
    mean=[0, 0, 0],
    std=[1, 1, 1]
)

input_tensor.requires_grad_(True)

tensor([[[[0.9098, 0.8667, 0.8196,  ..., 0.3451, 0.3490, 0.3647],
          [0.9059, 0.8784, 0.8431,  ..., 0.3490, 0.3529, 0.3686],
          [0.9020, 0.8980, 0.8745,  ..., 0.3569, 0.3608, 0.3765],
          ...,
          [0.2078, 0.2196, 0.2275,  ..., 0.4549, 0.4471, 0.4549],
          [0.2078, 0.2235, 0.2275,  ..., 0.4510, 0.4471, 0.4510],
          [0.2118, 0.2235, 0.2275,  ..., 0.4510, 0.4510, 0.4510]],

         [[0.7490, 0.7059, 0.6510,  ..., 0.2510, 0.2549, 0.2706],
          [0.7412, 0.7137, 0.6745,  ..., 0.2549, 0.2588, 0.2745],
          [0.7294, 0.7255, 0.7020,  ..., 0.2549, 0.2667, 0.2824],
          ...,
          [0.2588, 0.2784, 0.2824,  ..., 0.5608, 0.5608, 0.5608],
          [0.2627, 0.2745, 0.2784,  ..., 0.5608, 0.5647, 0.5647],
          [0.2667, 0.2745, 0.2784,  ..., 0.5647, 0.5647, 0.5647]],

         [[0.7725, 0.7294, 0.6784,  ..., 0.1412, 0.1451, 0.1608],
          [0.7686, 0.7412, 0.7020,  ..., 0.1451, 0.1490, 0.1647],
          [0.7608, 0.7569, 0.7294,  ..., 0

## 8. Generación del mapa Grad-CAM

Se genera el mapa de calor Grad-CAM utilizando el modelo adaptado, la capa objetivo y la clase seleccionada. El resultado indica las zonas de mayor activación asociadas a la predicción de la clase `robo`.

In [8]:
cam = GradCAM(
    model=wrapped_model,
    target_layers=target_layers
)

grayscale_cam = cam(
    input_tensor=input_tensor,
    targets=targets
)[0]

## 9. Visualización y guardado del resultado

El mapa de calor se superpone sobre la imagen original para visualizar las regiones que influyeron en la predicción del modelo. Finalmente, el resultado se guarda como imagen.

In [9]:
cam_image = show_cam_on_image(
    rgb_float,
    grayscale_cam,
    use_rgb=True
)

cam_image_bgr = cv2.cvtColor(cam_image, cv2.COLOR_RGB2BGR)

cv2.imwrite("gradcam_robo.jpg", cam_image_bgr)

cv2.imshow("Grad-CAM YOLO - Robo", cam_image_bgr)
cv2.waitKey(0)
cv2.destroyAllWindows()